In [ ]:

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load dataset
data = pd.read_csv('articles.csv')

# 2. Preprocess text
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', str(text))
    words = text.lower().split()
    words = [w for w in words if w not in stop_words]
    return words

sentences = data['text'].dropna().apply(clean_text).tolist()

# 3. Train CBOW model
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    workers=4
)

# 4. Convert documents into vectors
def document_vector(words):
    vectors = [model.wv[word] for word in words if word in model.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

doc_vectors = np.array([document_vector(doc) for doc in sentences])

# 5. Take user input
query = input("Enter your search text: ")
query_words = clean_text(query)

query_vector = document_vector(query_words)

# 6. Calculate cosine similarity
similarity = cosine_similarity(
    [query_vector], doc_vectors
)[0]

# 7. Recommend similar documents
indices = np.argsort(similarity)[::-1][:3]

print("Recommended Documents:")

for i in indices:
    print(data['text'].dropna().iloc[i])
    print("Similarity:", similarity[i])
    print()
